In [13]:
#TF-IDF Analyse für CV Runs

# -----------------
# Imports & Config
# -----------------

import re
import ast
import json
import os
import spacy
import numpy as np
import pandas as pd
from pathlib import Path
try:
    from stop_words import get_stop_words
except ImportError:
    from spacy.lang.de.stop_words import STOP_WORDS as _SPACY_GERMAN_STOP_WORDS

    def get_stop_words(language):
        if str(language).lower() != "german":
            return []
        return _SPACY_GERMAN_STOP_WORDS



from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline

# --- Warnings ---
import warnings
warnings.filterwarnings(
    "ignore",
    message="The parameter 'token_pattern' will not be used since 'tokenizer' is not None",
    category=UserWarning
)

# ------------------------------------------------------------
# Pfade & Config
# ------------------------------------------------------------


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "llm_pipeline").exists() and (candidate / "CV_analysis").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing llm_pipeline and CV_analysis.")


REPO_ROOT = find_repo_root()
RAW_RUNS_CSV = REPO_ROOT / "llm_pipeline/outputs_export/cv_runs.csv"
RUNS_CSV = REPO_ROOT / "llm_pipeline/outputs_export/cv_runs_fixed.csv"
NAMES_CSV = REPO_ROOT / "CV_analysis/data/CV_names_1985.csv"
NAME_LIST_DIR = REPO_ROOT / "CV_analysis/data/name_lists"

OUT_DIR = REPO_ROOT / "CV_analysis/data/tfidf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 30
QWEN_TOP_K_VARIANTS = [None, 64]


TOKEN_RE = re.compile(r"(?u)\b[^\W\d_]{2,}\b")

# vorher im Terminal: python -m spacy download de_core_news_sm
nlp = spacy.load("de_core_news_md")





In [14]:
# ------------------------------------------------------------
# Build/validate unified fixed CV runs
# ------------------------------------------------------------


def clean_response_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(v) for v in value]
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def parse_response_payload(value):
    text = clean_response_text(value)
    if not text:
        return None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", SyntaxWarning)
        for parser in (json.loads, ast.literal_eval):
            try:
                payload = parser(text)
                if isinstance(payload, (dict, list, tuple, set)):
                    return to_jsonable(payload)
            except Exception:
                pass

        if "{" in text and "}" in text:
            snippet = text[text.find("{"):text.rfind("}") + 1]
            for parser in (json.loads, ast.literal_eval):
                try:
                    payload = parser(snippet)
                    if isinstance(payload, (dict, list, tuple, set)):
                        return to_jsonable(payload)
                except Exception:
                    pass

    return None


def canonical_response_json(payload):
    return json.dumps(to_jsonable(payload), ensure_ascii=False, sort_keys=True)


def top_k_key(value):
    if pd.isna(value):
        return ""
    try:
        value_float = float(value)
        if value_float.is_integer():
            return str(int(value_float))
    except Exception:
        pass
    return str(value)


def fixed_row_key(row):
    key_cols = ["provider", "model", "profile_id", "first_name", "last_name"]
    return tuple(str(row.get(c, "")) for c in key_cols) + (top_k_key(row.get("top_k")),)


def ensure_runs_columns(df):
    for col in ["response_text", "top_p", "top_k"]:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def remove_personal_block_from_raw_text(value):
    text = clean_response_text(value)
    match = re.search(r'"01_persoenliche_daten"\s*:\s*', text)
    if not match:
        return text

    pos = match.end()
    while pos < len(text) and text[pos].isspace():
        pos += 1
    if pos >= len(text) or text[pos] != "{":
        return text

    in_string = False
    escape = False
    depth = 0
    block_end = None
    for i in range(pos, len(text)):
        char = text[i]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    block_end = i + 1
                    break

    if block_end is None:
        return text

    while block_end < len(text) and text[block_end].isspace():
        block_end += 1
    if block_end < len(text) and text[block_end] == ",":
        block_end += 1

    return (text[:match.start()] + text[block_end:]).strip()


def malformed_response_wrapper(row):
    raw_text = clean_response_text(row.get("response_text")) or clean_response_text(row.get("response_json"))
    return {
        "_repaired_from_malformed_json": True,
        "_repair_reason": "raw_response_json_and_response_text_unparseable",
        "raw_response_text_without_personal_data": remove_personal_block_from_raw_text(raw_text),
    }


def update_fixed_runs_csv(raw_path=RAW_RUNS_CSV, fixed_path=RUNS_CSV):
    raw = ensure_runs_columns(pd.read_csv(raw_path))
    existing = ensure_runs_columns(pd.read_csv(fixed_path)) if fixed_path.exists() else pd.DataFrame()

    existing_by_key = {}
    if not existing.empty:
        for _, row in existing.iterrows():
            payload = parse_response_payload(row.get("response_json"))
            if payload is not None:
                existing_by_key[fixed_row_key(row)] = canonical_response_json(payload)

    rows = []
    stats = []
    for _, row in raw.iterrows():
        fixed_row = row.copy()
        key = fixed_row_key(fixed_row)

        if key in existing_by_key:
            fixed_row["response_json"] = existing_by_key[key]
            status = "kept_existing_fixed"
        else:
            payload = parse_response_payload(fixed_row.get("response_json"))
            source = "response_json"
            if payload is None:
                payload = parse_response_payload(fixed_row.get("response_text"))
                source = "response_text"
            if payload is None:
                payload = malformed_response_wrapper(fixed_row)
                source = "valid_wrapper_for_malformed_response_text"

            fixed_row["response_json"] = canonical_response_json(payload)
            status = f"fixed_from_{source}"

        rows.append(fixed_row)
        stats.append((fixed_row.get("provider"), fixed_row.get("model"), fixed_row.get("top_k"), status))

    fixed = pd.DataFrame(rows)
    preferred_cols = [
        "model", "response_text", "provider", "age", "profile_id", "response_json",
        "top_p", "top_k", "first_name", "last_name",
        "prompt_tokens", "completion_tokens", "total_tokens",
    ]
    cols = [c for c in preferred_cols if c in fixed.columns] + [c for c in fixed.columns if c not in preferred_cols]
    fixed = fixed[cols]

    for response_json in fixed["response_json"]:
        json.loads(response_json)

    fixed.to_csv(fixed_path, index=False)
    stats_df = pd.DataFrame(stats, columns=["provider", "model", "top_k", "status"])
    return fixed, stats_df


df_runs_fixed_preview, fix_stats = update_fixed_runs_csv()
print(f"Saved {RUNS_CSV} with {df_runs_fixed_preview.shape[0]} valid rows.")
print(df_runs_fixed_preview.groupby(["provider", "model", "top_k"], dropna=False).size())
if not fix_stats.empty:
    print("\nFixed CSV repair statuses:")
    print(fix_stats.groupby(["provider", "model", "top_k", "status"], dropna=False).size())





Saved /Users/charlotteleininger/Master/Consulting/llm_pipeline/outputs_export/cv_runs_fixed.csv with 9600 valid rows.
provider       model                  top_k
google-gemini  gemini-2.5-flash-lite  NaN      1200
openai         gpt-4o-mini            NaN      1200
qwen3-14b      Qwen/Qwen3-14B         64.0     1200
                                      NaN      1200
qwen3-4b       Qwen/Qwen3-4B          64.0     1200
                                      NaN      1200
qwen3-8b       Qwen/Qwen3-8B          64.0     1200
                                      NaN      1200
dtype: int64

Fixed CSV repair statuses:
provider       model                  top_k  status             
google-gemini  gemini-2.5-flash-lite  NaN    kept_existing_fixed    1200
openai         gpt-4o-mini            NaN    kept_existing_fixed    1200
qwen3-14b      Qwen/Qwen3-14B         64.0   kept_existing_fixed    1200
                                      NaN    kept_existing_fixed    1200
qwen3-4b       Qwen/Qwen

In [15]:

#--------------------------------------------
# Helpers
#--------------------------------------------


def extract_all_text_without_personal(payload):
    """
    Entfernt '01_persoenliche_daten' komplett und extrahiert rekursiv Text.
    Bei Sprach-Sektionen ('sprache'/'sprachen') werden die Keys (z.B. 'Türkisch')
    zusätzlich als Text übernommen.
    """

    # persönlichen Block komplett entfernen
    if isinstance(payload, dict) and "01_persoenliche_daten" in payload:
        payload = {k: v for k, v in payload.items() if k != "01_persoenliche_daten"}

    texts = []

    def rec(x, parent_key=None):
        # dict
        if isinstance(x, dict):
            is_language_section = (
                parent_key is not None
                and isinstance(parent_key, str)
                and ("sprache" in parent_key.lower() or "sprachen" in parent_key.lower())
            )

            if is_language_section:
                for lang, level in x.items():
                    if isinstance(lang, str):
                        lang_clean = lang.strip()
                        if lang_clean:
                            if isinstance(level, str) and level.strip():
                                texts.append(f"{lang_clean} ({level.strip()})")
                            else:
                                texts.append(lang_clean)

            for k, v in x.items():
                rec(v, parent_key=k)

        elif isinstance(x, list):
            for v in x:
                rec(v, parent_key=parent_key)


        elif isinstance(x, str):
            t = x.strip()
            if t:
                texts.append(t)


    rec(payload)
    return "\n".join(texts)




def _name_parts(name):
    if not isinstance(name, str):
        return []
    parts = re.split(r"[\s\-]+", name.strip())
    return [p for p in parts if len(p) > 2]


def extract_name_from_json(payload):
    """
    Namen aus JSON ziehen (z. B. aus 'persoenliche_daten')
    """
    if not isinstance(payload, dict):
        return ""
    section = None

    # 1) Keys 'persoenliche_daten'
    for k, v in payload.items():
        if isinstance(v, dict) and (
            "persoenliche_daten" in k.lower()
            or "persönliche_daten" in k.lower()
            or "persoenliche" in k.lower()
        ):
            section = v
            break

    # 2)
    if section is None:
        for v in payload.values():
            if isinstance(v, dict) and any(kk.lower() == "name" for kk in v.keys()):
                section = v
                break

    if section is None:
        return ""

    for key in ["Name", "name", "voller_name", "vollname", "full_name"]:
        if key in section and isinstance(section[key], str) and section[key].strip():
            return section[key].strip()

    first = next((section.get(k) for k in ["Vorname", "vorname", "first_name"] if isinstance(section.get(k), str) and section.get(k).strip()), "")
    last = next((section.get(k) for k in ["Nachname", "nachname", "last_name"] if isinstance(section.get(k), str) and section.get(k).strip()), "")
    return " ".join(part.strip() for part in [first, last] if part and part.strip())


def mask_candidate_name_in_text(text, name_str):
    """
    Entfernt Vorkommen des Namens aus Text,
    falls wiederholt.
    """
    if not isinstance(text, str) or not name_str:
        return text

    parts = _name_parts(name_str)
    out = text
    for p in parts:
        pat = re.compile(r"\b" + re.escape(p) + r"\b", flags=re.IGNORECASE)
        out = pat.sub("", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out


def remove_numbers(text):
    """Entfernt Zahlen, Prozente, Datums-/Zeitangaben etc."""
    if not isinstance(text, str):
        return text
    patterns = [
        r"\b\d{1,2}:\d{2}\b",
        r"\b\d{2,4}[/-]\d{1,2}([/-]\d{1,2})?\b",
        r"\b\d{2,4}[/-]\d{2,4}\b",
        r"\b\d{2,4}\s*[–\-]\s*\d{2,4}\b",
        r"\b\d+(?:[.,]\d+)*\s*%\b",
        r"\b\d+(?:[.,]\d+)*\b",
    ]
    out = text
    for pat in patterns:
        out = re.sub(pat, " ", out)
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out


INFLECTION_NORMALIZATION = {
    # spaCy sometimes leaves these adjective endings in place.
    "erfahrene": "erfahren",
    "erfahrener": "erfahren",
    "erfahrenes": "erfahren",
    "erfahrenen": "erfahren",
    "erfahrenem": "erfahren",
    "erfahrenе": "erfahren",  # final character is occasionally Cyrillic e in model output
    "erfahrend": "erfahren",
    # These are not gender markers; keep the underlying descriptor/professional noun.
    "kommunikationsstarke": "kommunikationsstark",
    "kommunikationsstarker": "kommunikationsstark",
    "kommunikationsstarkes": "kommunikationsstark",
    "kommunikationsstarken": "kommunikationsstark",
    "kommunikationsstarkem": "kommunikationsstark",
    "kommunikationsprofis": "kommunikationsprofi",
}

ADJECTIVE_STEMS_TO_DEINFLECT = (
    "erfahren",
    "berufserfahren",
    "kommunikationsstark",
    "ergebnisorientiert",
)


def normalize_inflected_descriptors(token: str) -> str:
    """Normalize frequent adjective/noun inflections that spaCy leaves as separate terms."""
    if not isinstance(token, str):
        return token

    t = token.lower().replace("е", "e")  # normalize Cyrillic e lookalike
    if t in INFLECTION_NORMALIZATION:
        return INFLECTION_NORMALIZATION[t]

    for stem in ADJECTIVE_STEMS_TO_DEINFLECT:
        if t == stem:
            return stem
        for ending in ("e", "er", "es", "en", "em"):
            if t == stem + ending:
                return stem

    return t


def normalize_gender_forms(token: str) -> str:
    """
    Normalisiert deutsche Genderformen in Tokens.
    """
    if not isinstance(token, str):
        return token

    t = normalize_inflected_descriptors(token)

    if t.endswith("keite"):
        t = t[:-1]

    if t.startswith("berufserfahren"):
        return "berufserfahren"

    # generische Gender-Normalisierung
    if t.endswith("innen") and len(t) > 6:
        t = t[:-5]      # 'beraterinnen' -> 'berater'
    elif t.endswith("in") and len(t) > 4:
        t = t[:-2]      # 'beraterin' -> 'berater'

    t = normalize_inflected_descriptors(t)

    # Spezialfall: fachfrau/fachmann -> fachkraft
    if t.endswith("frau") and len(t) > 6:
        return "fachkraft" if t == "fachfrau" else t[:-4] + "-mann/-frau"

    if t.endswith("mann") and len(t) > 6:
        return "fachkraft" if t == "fachmann" else t[:-4] + "-mann/-frau"

    if t.endswith("experte"):
        root = t[:-1]   # alles vor 'experte'
        return root + "e/-in"

    if t.endswith("expert"):
        root = t
        return root + "e/-in"

    return t


def row_mean_as_array(X_subset):
    m = X_subset.mean(axis=0)
    if hasattr(m, "A1"):
        return m.A1
    return np.asarray(m).ravel()




def load_name_stopwords(names_csv=NAMES_CSV, name_list_dir=NAME_LIST_DIR):
    name_words = set()
    paths = [names_csv]
    if name_list_dir.exists():
        paths.extend(sorted(name_list_dir.glob("*.csv")))

    for path in paths:
        if not path.exists():
            continue
        df_names_stop = pd.read_csv(path)
        for col in df_names_stop.columns:
            if "name" not in col.lower():
                continue
            for value in df_names_stop[col].dropna().astype(str):
                name_words.update(part.lower() for part in _name_parts(value))
    return name_words


german_stop = set(
    w.lower() for w in get_stop_words("german")
    if isinstance(w, str) and w.strip()
)
name_stop = load_name_stopwords()




In [16]:
def gender_tokenize(text):
    if not isinstance(text, str):
        return []
    doc = nlp(text)
    out = []
    for token in doc:
        if token.is_space or token.is_punct:
            continue

        raw_text = token.text.lower()
        lemma = token.lemma_.lower()
        if not lemma:
            continue

        # Fix: Muttersprache
        if len(lemma) < len(raw_text) and raw_text.endswith("sprache"):
            lemma = raw_text

        # Spezialregeln für Manager/Managerin da nicht korrekt gelemmatized
        if raw_text.endswith("managerin"):
            lemma = raw_text[:-2]
        elif raw_text.endswith("manager"):
            lemma = raw_text

        # Sales bleibt sales, da nicht english und nicht gelemmatized
        if raw_text == "sales":
            lemma = "sales"

        # --- Sonderregel nur für 'ergebnisorientierter' ---
        if raw_text.startswith("ergebnisorientiert"):
            lemma = "ergebnisorientiert"


        if not TOKEN_RE.fullmatch(lemma):
            continue

        if lemma in german_stop:
            continue
        if lemma in name_stop:
            continue

        t_norm = normalize_gender_forms(lemma)
        if t_norm in name_stop:
            continue
        if t_norm:
            out.append(t_norm)

    return out



In [7]:


print(os.getcwd())


/Users/charlotteleininger/Master/Consulting/CV_analysis/code/Tfidf


In [5]:
os.chdir("/Users/charlotteleininger/Master/Consulting")


In [17]:
# Mappings laden (gender/ethnicity + name_ID)
df_names = pd.read_csv(NAMES_CSV)
df_names["first_name_lower"] = df_names["first_name"].str.strip().str.lower()
df_names["last_name_lower"]  = df_names["last_name"].str.strip().str.lower()

df_runs = pd.read_csv(RUNS_CSV)
df_runs["first_name_lower"] = df_runs["first_name"].str.strip().str.lower()
df_runs["last_name_lower"]  = df_runs["last_name"].str.strip().str.lower()

df = df_runs.merge(
    df_names,
    on=["first_name_lower", "last_name_lower"],
    how="left",
    suffixes=("", "_names"),
)


def model_subset(provider_pattern, top_k=None):
    provider_mask = df["provider"].astype(str).str.contains(provider_pattern, case=False, na=False)
    if top_k is None:
        top_k_mask = df["top_k"].isna() if "top_k" in df.columns else True
    else:
        top_k_mask = pd.to_numeric(df["top_k"], errors="coerce").eq(top_k)
    return df[provider_mask & top_k_mask].copy()


df_openai = model_subset("openai")
df_gemini = model_subset("google-gemini")
df_qwen_4B = model_subset("qwen3-4b")
df_qwen_8B = model_subset("qwen3-8b")
df_qwen_14B = model_subset("qwen3-14b")
df_qwen_4B_topk64 = model_subset("qwen3-4b", top_k=64)
df_qwen_8B_topk64 = model_subset("qwen3-8b", top_k=64)
df_qwen_14B_topk64 = model_subset("qwen3-14b", top_k=64)

print("Model input rows:")
for label, runs_df in {
    "chatgpt": df_openai,
    "gemini": df_gemini,
    "qwen4B": df_qwen_4B,
    "qwen8B": df_qwen_8B,
    "qwen14B": df_qwen_14B,
    "qwen4B_topk64": df_qwen_4B_topk64,
    "qwen8B_topk64": df_qwen_8B_topk64,
    "qwen14B_topk64": df_qwen_14B_topk64,
}.items():
    print(f"{label}: {runs_df.shape[0]}")


def prepare_docs_from_runs(df, mask_numbers=True):
    texts = []
    gender_labels = []
    ethnicity_labels = []
    profile_ids = []
    name_ids = []

    for idx, row in df.iterrows():
        raw_json = row.get("response_json", "")
        profile_id = row.get("profile_id")
        gender = row.get("gender")
        ethnicity = row.get("ethnicity")
        name_id = row.get("name_ID")

        payload = parse_response_payload(raw_json)
        if payload is None:
            continue

        fulltext = extract_all_text_without_personal(payload)

        json_name = extract_name_from_json(payload)
        if json_name:
            fulltext = mask_candidate_name_in_text(fulltext, json_name)
        fulltext = mask_candidate_name_in_text(fulltext, row.get("first_name"))
        fulltext = mask_candidate_name_in_text(fulltext, row.get("last_name"))

        if mask_numbers:
            fulltext = remove_numbers(fulltext)

        texts.append(fulltext)
        gender_labels.append(gender)
        ethnicity_labels.append(ethnicity)
        profile_ids.append(profile_id)
        name_ids.append(name_id)

    df_docs = pd.DataFrame({
        "text": texts,
        "gender": gender_labels,
        "ethnicity": ethnicity_labels,
        "profile_id": profile_ids,
        "name_ID": name_ids,
    })

    return df_docs




Model input rows:
chatgpt: 1200
gemini: 1200
qwen4B: 1200
qwen8B: 1200
qwen14B: 1200
qwen4B_topk64: 1200
qwen8B_topk64: 1200
qwen14B_topk64: 1200


In [18]:

def make_tfidf_vectorizer(use_idf=True):
    return TfidfVectorizer(
        tokenizer=gender_tokenize,
        lowercase=True,
        ngram_range=(1, 1),
        min_df=10,
        max_df=0.95,
        use_idf=use_idf,
        norm=None,
        smooth_idf=False,
    )


def sparse_mean_1d(X):
    m = X.mean(axis=0)
    return m.A1 if hasattr(m, "A1") else np.asarray(m).ravel()


def sparse_sum_1d(X):
    s = X.sum(axis=0)
    return s.A1 if hasattr(s, "A1") else np.asarray(s).ravel()


def profile_level_ate_report(
    X_tfidf,
    X_tf,
    terms,
    labels,
    label_a,
    label_b,
    top_k=30,
    model_name=None,
    contrast=None,
    sort_by="ATE_tfidf",
    ascending=False,
):
    """
    Average treatment effect style report for two groups.

    ATE_tfidf is mean(tf-idf | label_a) - mean(tf-idf | label_b).
    Positive values are terms used more by label_a; negative values are terms used more by label_b.
    X_tf is the matching raw term-frequency matrix from the same vocabulary/tokenizer.
    """
    labels = np.asarray(labels)
    terms = np.asarray(terms)

    if X_tfidf.shape != X_tf.shape:
        raise ValueError("X_tfidf and X_tf must have the same shape and vocabulary.")
    if X_tfidf.shape[1] != len(terms):
        raise ValueError("Number of terms does not match matrix columns.")

    mask = (labels == label_a) | (labels == label_b)
    if mask.sum() == 0:
        raise ValueError(f"No rows found for {label_a} or {label_b}.")

    Xf = X_tfidf[mask]
    Xt = X_tf[mask]
    lbl = labels[mask]
    y = (lbl == label_a).astype(int)

    if y.sum() == 0 or (y == 0).sum() == 0:
        raise ValueError(f"Both groups must be present: {label_a}, {label_b}.")

    chi2_scores, chi2_p = chi2(Xf, y)

    Xa_f = Xf[y == 1]
    Xb_f = Xf[y == 0]
    Xa_t = Xt[y == 1]
    Xb_t = Xt[y == 0]

    mu_a_tfidf = sparse_mean_1d(Xa_f)
    mu_b_tfidf = sparse_mean_1d(Xb_f)
    sum_a_tf = sparse_sum_1d(Xa_t)
    sum_b_tf = sparse_sum_1d(Xb_t)

    ate = mu_a_tfidf - mu_b_tfidf
    eps = 1e-6
    tf_ratio = (sum_a_tf + eps) / (sum_b_tf + eps)
    doc_share_a = np.asarray((Xa_t > 0).mean(axis=0)).ravel()
    doc_share_b = np.asarray((Xb_t > 0).mean(axis=0)).ravel()

    df = pd.DataFrame({
        "model": model_name,
        "contrast": contrast or f"{label_a}_vs_{label_b}",
        "label_a": label_a,
        "label_b": label_b,
        "term": terms,
        f"avg_tfidf_{label_a}": mu_a_tfidf,
        f"avg_tfidf_{label_b}": mu_b_tfidf,
        "ATE_tfidf": ate,
        f"sum_tf_{label_a}": sum_a_tf,
        f"sum_tf_{label_b}": sum_b_tf,
        "tf_ratio_a_over_b": tf_ratio,
        f"doc_share_{label_a}": doc_share_a,
        f"doc_share_{label_b}": doc_share_b,
        "chi2": chi2_scores,
        "chi2_p": chi2_p,
    })

    if sort_by == "abs_ATE_tfidf":
        df = df.assign(abs_ATE_tfidf=df["ATE_tfidf"].abs()).sort_values(
            "abs_ATE_tfidf", ascending=ascending
        )
    else:
        df = df.sort_values(sort_by, ascending=ascending)

    out = df.head(top_k).copy()

    print(f"\n{model_name or ''} {contrast or ''}: top {top_k} terms for {label_a} vs {label_b}")
    display_cols = [
        "term", "ATE_tfidf", f"avg_tfidf_{label_a}", f"avg_tfidf_{label_b}",
        f"sum_tf_{label_a}", f"sum_tf_{label_b}", "tf_ratio_a_over_b", "chi2", "chi2_p"
    ]
    print(out[display_cols].round(4).to_string(index=False))
    return out


def term_tf_by_group(
    X_tf,
    terms,
    labels,
    label_a,
    label_b,
    target_terms,
):
    labels = np.asarray(labels)
    terms = np.asarray(terms)
    mask = (labels == label_a) | (labels == label_b)
    Xt = X_tf[mask]
    lbl = labels[mask]
    term2idx = {t: i for i, t in enumerate(terms)}
    rows = []

    for term in target_terms:
        if term not in term2idx:
            rows.append({
                "term": term,
                f"sum_tf_{label_a}": np.nan,
                f"sum_tf_{label_b}": np.nan,
                "tf_ratio_a_over_b": np.nan,
            })
            continue

        col = Xt[:, term2idx[term]].toarray().ravel()
        sum_a = col[lbl == label_a].sum()
        sum_b = col[lbl == label_b].sum()
        rows.append({
            "term": term,
            f"sum_tf_{label_a}": sum_a,
            f"sum_tf_{label_b}": sum_b,
            "tf_ratio_a_over_b": (sum_a + 1e-6) / (sum_b + 1e-6),
        })

    df = pd.DataFrame(rows)
    print(f"\nTF summary for selected terms ({label_a} vs {label_b})")
    print(df.to_string(index=False))
    return df



In [8]:
# Manuelle suche nach Wortvorkommen im DataFrame

def inspect_word_occurrences(
    df,
    query="türkisch",
    text_col=None,
    n_examples=None,
    partial=True
):
    """
    Sucht nach query in einer Textspalte eines DataFrames, zeigt Kontexte
    und einfache Stats nach gender/ethnicity, wenn vorhanden.

    n_examples:
      - None  -> alle Kontexte ausgeben
      - int   -> nur die ersten n_examples ausgeben
    """
    if text_col is None:
        if "response_json" in df.columns:
            text_col = "response_json"
        elif "text" in df.columns:
            text_col = "text"
        else:
            raise ValueError(
                "Kein text_col angegeben und weder 'response_json' noch 'text' im DataFrame gefunden."
            )

    q = str(query).strip()
    if not q:
        raise ValueError("query ist leer.")

    # Phrase-Regex bauen (tokenweise)
    tokens = re.findall(r"\w+", q.lower())
    if partial:
        token_pats = [rf"\w*{re.escape(t)}\w*" for t in tokens]
    else:
        token_pats = [rf"\b{re.escape(t)}\b" for t in tokens]

    phrase_pat = r"(?:\W+)".join(token_pats)

    # Kontext: bis zu 3 Wörter davor und danach
    context_re = re.compile(
        rf"(?:\w+\W+){{0,3}}({phrase_pat})(?:\W+\w+){{0,3}}",
        flags=re.IGNORECASE
    )

    contains_mask = df[text_col].astype(str).str.contains(
        phrase_pat, regex=True, na=False, case=False
    )
    hits = df[contains_mask].copy()

    print(f"\nProfiles with query '{q}': {len(hits)}")

    contexts = []
    for _, row in hits.iterrows():
        txt = str(row[text_col])
        m = context_re.search(txt)
        contexts.append(m.group(0).strip() if m else "")
    hits["context"] = contexts

    print("\nSample contexts:")
    cols_to_show = [c for c in ["profile_id", "gender", "ethnicity", "context"] if c in hits.columns]

    to_show = hits if n_examples is None else hits.head(n_examples)

    if cols_to_show:
        for _, row in to_show[cols_to_show].iterrows():
            meta = " | ".join(str(row[c]) for c in cols_to_show[:-1])
            print(f"{meta} -> {row['context']}")
    else:
        for ctx in to_show["context"]:
            print("->", ctx)

    if "gender" in hits.columns:
        print("\nGender Count:")
        print(hits["gender"].value_counts())

    if "ethnicity" in hits.columns:
        print("\nEthnicity Count:")
        print(hits["ethnicity"].value_counts())

        for eth in ["german", "turkish"]:
            subset = hits[hits["ethnicity"] == eth]
            if not subset.empty:
                print(f"\nHits for ethnicity == '{eth}':")
                cols = [
                    c for c in
                    ["profile_id", "name_ID", "first_name", "last_name", "context"]
                    if c in subset.columns
                ]
                print(subset[cols])

    return hits



In [19]:

# --------------------------------------------------------
# Build TF / TF-IDF matrices for selected CV models
# --------------------------------------------------------

MODEL_INPUTS = {
    "chatgpt": df_openai,
    "gemini": df_gemini,
    "qwen4B": df_qwen_4B,
    "qwen8B": df_qwen_8B,
    "qwen14B": df_qwen_14B,
    "qwen4B_topk64": df_qwen_4B_topk64,
    "qwen8B_topk64": df_qwen_8B_topk64,
    "qwen14B_topk64": df_qwen_14B_topk64,
}

RAW_TFIDF_PATHS = {
    "chatgpt": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_chatgpt.csv",
    "gemini": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_gem.csv",
    "qwen4B": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen4B.csv",
    "qwen8B": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen8B.csv",
    "qwen14B": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen14B.csv",
    "qwen4B_topk64": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen4B_topk64.csv",
    "qwen8B_topk64": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen8B_topk64.csv",
    "qwen14B_topk64": REPO_ROOT / "CV_analysis/data/raw_tfidf_traintest_qwen14B_topk64.csv",
}

# Choose which models to regenerate in this cell.
# Set to None to run all models.
TFIDF_MODELS_TO_RUN = [
    "qwen4B_topk64",
    "qwen8B_topk64",
    "qwen14B",
    "qwen14B_topk64",
]

if TFIDF_MODELS_TO_RUN is None:
    selected_model_names = list(MODEL_INPUTS.keys())
else:
    unknown_models = sorted(set(TFIDF_MODELS_TO_RUN) - set(MODEL_INPUTS.keys()))
    if unknown_models:
        raise ValueError(f"Unknown model keys in TFIDF_MODELS_TO_RUN: {unknown_models}")
    selected_model_names = list(TFIDF_MODELS_TO_RUN)

print("Regenerating TF-IDF for:", selected_model_names)

model_tfidf = {}

for model_name in selected_model_names:
    runs_df = MODEL_INPUTS[model_name]
    docs = prepare_docs_from_runs(runs_df, mask_numbers=True).dropna(subset=["gender"]).copy()
    if docs.empty:
        print(f"{model_name}: no valid docs, skipping")
        continue

    tfidf_vec = make_tfidf_vectorizer(use_idf=True)
    tf_vec = make_tfidf_vectorizer(use_idf=False)

    X_tfidf = tfidf_vec.fit_transform(docs["text"])
    terms = tfidf_vec.get_feature_names_out()

    # Reuse the TF-IDF vocabulary for raw TF so both matrices align column-for-column.
    tf_vec.set_params(vocabulary=tfidf_vec.vocabulary_)
    X_tf = tf_vec.fit_transform(docs["text"])
    terms_tf = tf_vec.get_feature_names_out()
    if list(terms) != list(terms_tf):
        raise ValueError(f"TF and TF-IDF vocabularies differ for {model_name}.")

    tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=terms)
    tfidf_df.insert(0, "profile_id", docs["profile_id"].values)
    tfidf_df.insert(1, "gender", docs["gender"].values)
    tfidf_df.insert(2, "ethnicity", docs["ethnicity"].values)
    tfidf_df.insert(3, "name_ID", docs["name_ID"].values)
    tfidf_df.to_csv(RAW_TFIDF_PATHS[model_name], index=False)

    model_tfidf[model_name] = {
        "docs": docs,
        "X_tfidf": X_tfidf,
        "X_tf": X_tf,
        "terms": terms,
        "vectorizer": tfidf_vec,
        "raw_tfidf_path": RAW_TFIDF_PATHS[model_name],
    }

    print(f"{model_name}: docs={docs.shape[0]}, tfidf_shape={X_tfidf.shape}, saved={RAW_TFIDF_PATHS[model_name]}")

# Backwards-compatible names for later exploratory cells, only for models run in this session.
if "chatgpt" in model_tfidf:
    df_gender_cv_chat = model_tfidf["chatgpt"]["docs"]
    X_gcv_chat = model_tfidf["chatgpt"]["X_tfidf"]
    X_tf_chat = model_tfidf["chatgpt"]["X_tf"]
    terms_gcv_chat = model_tfidf["chatgpt"]["terms"]
    terms_tf_chat = model_tfidf["chatgpt"]["terms"]
    df_docs_chatgpt = df_gender_cv_chat
    df_eth_cv_chat = df_gender_cv_chat.dropna(subset=["ethnicity"]).copy()

if "gemini" in model_tfidf:
    df_gender_cv_gem = model_tfidf["gemini"]["docs"]
    X_gcv_gem = model_tfidf["gemini"]["X_tfidf"]
    X_tf_gem = model_tfidf["gemini"]["X_tf"]
    terms_gcv_gem = model_tfidf["gemini"]["terms"]
    terms_tf_gem = model_tfidf["gemini"]["terms"]
    df_docs_gemini = df_gender_cv_gem
    df_eth_cv_gem = df_gender_cv_gem.dropna(subset=["ethnicity"]).copy()

if "qwen4B" in model_tfidf:
    df_gender_cv_qwen_4B = model_tfidf["qwen4B"]["docs"]
    X_tfidf_qwen_4B = model_tfidf["qwen4B"]["X_tfidf"]
    X_tf_qwen_4B = model_tfidf["qwen4B"]["X_tf"]
    terms_tfidf_qwen_4B = model_tfidf["qwen4B"]["terms"]

if "qwen8B" in model_tfidf:
    df_gender_cv_qwen_8B = model_tfidf["qwen8B"]["docs"]
    X_tfidf_qwen_8B = model_tfidf["qwen8B"]["X_tfidf"]
    X_tf_qwen_8B = model_tfidf["qwen8B"]["X_tf"]
    terms_tfidf_qwen_8B = model_tfidf["qwen8B"]["terms"]

if "qwen14B" in model_tfidf:
    df_gender_cv_qwen_14B = model_tfidf["qwen14B"]["docs"]
    X_tfidf_qwen_14B = model_tfidf["qwen14B"]["X_tfidf"]
    X_tf_qwen_14B = model_tfidf["qwen14B"]["X_tf"]
    terms_tfidf_qwen_14B = model_tfidf["qwen14B"]["terms"]

# Alias used by older exploratory leakage cells below.
gender_norm_tokenize = gender_tokenize



Regenerating TF-IDF for: ['qwen4B_topk64', 'qwen8B_topk64', 'qwen14B', 'qwen14B_topk64']
qwen4B_topk64: docs=1200, tfidf_shape=(1200, 1352), saved=/Users/charlotteleininger/Master/Consulting/CV_analysis/data/raw_tfidf_traintest_qwen4B_topk64.csv
qwen8B_topk64: docs=1200, tfidf_shape=(1200, 1641), saved=/Users/charlotteleininger/Master/Consulting/CV_analysis/data/raw_tfidf_traintest_qwen8B_topk64.csv
qwen14B: docs=1200, tfidf_shape=(1200, 1939), saved=/Users/charlotteleininger/Master/Consulting/CV_analysis/data/raw_tfidf_traintest_qwen14B.csv
qwen14B_topk64: docs=1200, tfidf_shape=(1200, 1917), saved=/Users/charlotteleininger/Master/Consulting/CV_analysis/data/raw_tfidf_traintest_qwen14B_topk64.csv


In [11]:

# --------------------------------------------------------
# ATE reports for all models
# --------------------------------------------------------

ATE_OUT_DIR = REPO_ROOT / "CV_analysis/data/tfidf_ate"
ATE_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTRASTS = {
    "gender_female_vs_male": ("gender", "female", "male"),
    "gender_male_vs_female": ("gender", "male", "female"),
    "ethnicity_german_vs_turkish": ("ethnicity", "german", "turkish"),
    "ethnicity_turkish_vs_german": ("ethnicity", "turkish", "german"),
}

ate_reports = {}

for model_name, obj in model_tfidf.items():
    docs = obj["docs"]
    model_reports = []

    for contrast_name, (label_col, label_a, label_b) in CONTRASTS.items():
        report = profile_level_ate_report(
            X_tfidf=obj["X_tfidf"],
            X_tf=obj["X_tf"],
            terms=obj["terms"],
            labels=docs[label_col],
            label_a=label_a,
            label_b=label_b,
            top_k=30,
            model_name=model_name,
            contrast=contrast_name,
            sort_by="ATE_tfidf",
            ascending=False,
        )
        ate_reports[(model_name, contrast_name)] = report
        model_reports.append(report)
        report.to_csv(ATE_OUT_DIR / f"tfidf_ate_{model_name}_{contrast_name}.csv", index=False)

    pd.concat(model_reports, ignore_index=True).to_csv(
        ATE_OUT_DIR / f"tfidf_ate_{model_name}_all_contrasts.csv",
        index=False,
    )

all_ate_reports = pd.concat(ate_reports.values(), ignore_index=True)
all_ate_reports.to_csv(ATE_OUT_DIR / "tfidf_ate_all_models_all_contrasts.csv", index=False)
print(f"Saved ATE reports to {ATE_OUT_DIR}")




chatgpt gender_female_vs_male: top 30 terms for female vs male
                    term  ATE_tfidf  avg_tfidf_female  avg_tfidf_male  sum_tf_female  sum_tf_male  tf_ratio_a_over_b     chi2  chi2_p
               fachkraft     1.4878            1.5663          0.0785          459.0         23.0            19.9565 807.4915  0.0000
         kauf-mann/-frau     0.4956            0.5017          0.0061           82.0          1.0            81.9999 290.2046  0.0000
                    weit     0.2412            1.1588          0.9176          245.0        194.0             1.2629  16.8135  0.0000
                    frau     0.1849            0.3877          0.2028           65.0         34.0             1.9118  34.7437  0.0000
         sprachanwendung     0.1827            1.9064          1.7237          480.0        434.0             1.1060   5.5168  0.0188
                    ziel     0.1728            0.5473          0.3745          114.0         78.0             1.4615  19.4440  0.000

In [12]:

# --------------------------------------------------------
# Compact ATE overview tables
# --------------------------------------------------------

ate_overview = (
    all_ate_reports
    .sort_values(["model", "contrast", "ATE_tfidf"], ascending=[True, True, False])
    .groupby(["model", "contrast"], as_index=False)
    .head(10)
    .loc[:, ["model", "contrast", "term", "ATE_tfidf", "chi2", "chi2_p"]]
)

print("Top 10 ATE terms per model and contrast")
display(ate_overview.round({"ATE_tfidf": 4, "chi2": 2, "chi2_p": 4}))

ate_terms_wide = (
    all_ate_reports
    .sort_values(["model", "contrast", "ATE_tfidf"], ascending=[True, True, False])
    .groupby(["model", "contrast"])["term"]
    .apply(lambda terms: ", ".join(terms.head(10)))
    .reset_index(name="top_terms")
)

print("\nTop terms as one-row-per-model/contrast overview")
display(ate_terms_wide)



Top 10 ATE terms per model and contrast


,model,contrast,term,ATE_tfidf,chi2,chi2_p
60,chatgpt,ethnicity_german_vs_turkish,chancengerechtigkeit,0.2056,30.84,0.0000
61,chatgpt,ethnicity_german_vs_turkish,neu,0.1682,8.75,0.0031
62,chatgpt,ethnicity_german_vs_turkish,digital,0.1556,2.26,0.1329
63,chatgpt,ethnicity_german_vs_turkish,mitarbeit,0.1520,28.80,0.0000
64,chatgpt,ethnicity_german_vs_turkish,bildungsbereich,0.1504,25.18,0.0000
...,...,...,...,...,...,...
755,qwen8B_topk64,gender_male_vs_female,quereinstieg,0.5550,109.46,0.0000
756,qwen8B_topk64,gender_male_vs_female,webdesign,0.5514,123.36,0.0000
757,qwen8B_topk64,gender_male_vs_female,steuerwesen,0.5060,111.86,0.0000
758,qwen8B_topk64,gender_male_vs_female,kenntniss,0.4580,130.57,0.0000



Top terms as one-row-per-model/contrast overview


,model,contrast,top_terms
0,chatgpt,ethnicity_german_vs_turkish,"chancengerechtigkeit, neu, digital, mitarbeit,..."
1,chatgpt,ethnicity_turkish_vs_german,"sprachanwendung, annähernd, muttersprache, str..."
2,chatgpt,gender_female_vs_male,"fachkraft, kauf-mann/-frau, weit, frau, sprach..."
3,chatgpt,gender_male_vs_female,"fach-mann/-frau, sport, einstieg, consultant, ..."
4,gemini,ethnicity_german_vs_turkish,"musterstadt, verhandlungssich, muttersprache, ..."
5,gemini,ethnicity_turkish_vs_german,"jjjj, mm, professional, bank, deutschland, ken..."
6,gemini,gender_female_vs_male,"fachkraft, gut, kauf-mann/-frau, veranstaltung..."
7,gemini,gender_male_vs_female,"profi, beispiel, berufseinsteiger, berufstätig..."
8,qwen14B,ethnicity_german_vs_turkish,"muttersprache, fortgeschritten, sales, fach-ma..."
9,qwen14B,ethnicity_turkish_vs_german,"muttersprachlich, berl, international, kenntni..."


In [ ]:
import re

# Alle "Wörter", in denen irgendwo / oder * vorkommt.
# \S* = alles ohne Whitespace, d.h. wir schneiden später noch Satzzeichen weg
SPECIAL_TOKEN_RE = re.compile(r"(?u)\b\S*[/*]\S*\b")

def find_special_tokens(df, text_col="text"):
    tokens = set()
    for txt in df[text_col].dropna():
        txt = str(txt)
        matches = SPECIAL_TOKEN_RE.findall(txt)
        for m in matches:
            # Satzzeichen am Rand entfernen, damit "geehrte/r," -> "geehrte/r" wird
            m_clean = m.strip(".,;:!?()[]{}\"'„“‚’")
            if m_clean:
                tokens.add(m_clean)
    return tokens

# Für df_docs_chat
special_chat = find_special_tokens(df_docs_chatgpt, text_col="text")
print("Anzahl spezieller Tokens (ChatGPT):", len(special_chat))
print("Tokens (ChatGPT):")
for t in sorted(special_chat):
    print(t)

# Für df_docs_gemini
special_gemini = find_special_tokens(df_docs_gemini, text_col="text")
print("\nAnzahl spezieller Tokens (Gemini):", len(special_gemini))
print("Tokens (Gemini):")
for t in sorted(special_gemini):
    print(t)



In [ ]:
import re

# Zahl vor "Jahre Berufserfahrung" oder "jähriger Berufserfahrung"
RE_YEARS_EXP = re.compile(
    r"(\d{1,2}(?:[.,]\d+)?)\s*"          # 1–2-stellige Zahl, optional Komma/ Punkt
    r"(?:jahre?n?|jährig(?:e|er|en)?)\s+"  # jahr / jahre / jährig / jähriger / jährigen ...
    r"berufserfahrung",                  # Berufserfahrung
    flags=re.IGNORECASE
)

def extract_years_experience(text):
    """
    Extrahiert die (maximale) Anzahl an Jahren Berufserfahrung aus einem Text.
    Gibt float oder None zurück.
    """
    if not isinstance(text, str):
        return None

    matches = RE_YEARS_EXP.findall(text)
    if not matches:
        return None

    values = []
    for m in matches:
        s = m.replace(",", ".")
        try:
            values.append(float(s))
        except ValueError:
            continue

    if not values:
        return None

    # falls mehrere Vorkommen im Text: größte Zahl zurückgeben
    return max(values)

df_gender_cv_chat = df_gender_cv_chat.copy()
df_gender_cv_chat["years_exp"] = df_gender_cv_chat["text"].apply(extract_years_experience)

# Count NA vs non-NA
num_na = df_gender_cv_chat["years_exp"].isna().sum()
num_non_na = df_gender_cv_chat["years_exp"].notna().sum()

print("years_exp missing:", num_na)
print("years_exp non-missing:", num_non_na)

# Overall average per gender
avg_by_gender = (
    df_gender_cv_chat
    .groupby("gender")["years_exp"]
    .mean()
)

print("\nOverall avg years by gender:")
print(avg_by_gender)

# Per-profile average per gender
per_profile = (
    df_gender_cv_chat
    .groupby(["profile_id", "gender"])["years_exp"]
    .mean()
    .unstack("gender")
)

print("\nPer-profile avg years (female vs male):")
print(per_profile.to_string())





In [ ]:
# Overall average per ethnicity
avg_by_eth = (
    df_gender_cv_chat
    .groupby("ethnicity")["years_exp"]
    .mean()
)

print("\nOverall avg years by ethnicity:")
print(avg_by_eth)

# Per-profile average per ethnicity
per_profile = (
    df_gender_cv_chat
    .groupby(["profile_id", "ethnicity"])["years_exp"]
    .mean()
    .unstack("ethnicity")
)

print("\nPer-profile avg years by ethnicity:")
print(per_profile.to_string())



In [ ]:
df_gender_cv_gem = df_gender_cv_gem.copy()
df_gender_cv_gem["years_exp"] = df_gender_cv_gem["text"].apply(extract_years_experience)

# Count NA vs non-NA
num_na = df_gender_cv_gem["years_exp"].isna().sum()
num_non_na = df_gender_cv_gem["years_exp"].notna().sum()

print("years_exp missing:", num_na)
print("years_exp non-missing:", num_non_na)

# Overall average per gender
avg_by_gender_gem = (
    df_gender_cv_gem
    .groupby("gender")["years_exp"]
    .mean()
)

print("\nOverall avg years by gender:")
print(avg_by_gender_gem)

# Per-profile average per gender
per_profile = (
    df_gender_cv_gem
    .groupby(["profile_id", "gender"])["years_exp"]
    .mean()
    .unstack("gender")
)

print("\nPer-profile avg years (female vs male):")
print(per_profile.to_string())





In [ ]:
# Overall average per ethnicity
avg_by_eth_gem = (
    df_gender_cv_gem
    .groupby("ethnicity")["years_exp"]
    .mean()
)

print("\nOverall avg years by ethnicity:")
print(avg_by_eth_gem)

# Per-profile average per ethnicity
per_profile = (
    df_gender_cv_gem
    .groupby(["profile_id", "ethnicity"])["years_exp"]
    .mean()
    .unstack("ethnicity")
)

print("\nPer-profile avg years by ethnicity:")
print(per_profile.to_string())

